
#### Requisitos para o arquivo estadios.csv

Objetivo: realizar um processo de ETL e disponibilizar os dados em uma camada apropriada

Ler -> Transformar -> Escrever (ETL)

1. Ler o arquivo csv do Data Lake
2. Renomear as colunas para português utilizando o **snake case**
3. Definir o esquema de dados correto
4. Precisa ter uma coluna com a data em que o arquivo foi consumido
5. Salvar os dados em formato **parquet** na camada adequada


#### 1. Lendo o arquivo CSV

In [0]:
%run "../Modulo 2/Configurando o storage"

In [0]:
caminho_bronze = "abfss://bronze@datalakexperiun.dfs.core.windows.net/"
caminho_silver = "abfss://silver@datalakexperiun.dfs.core.windows.net/"
caminho_gold = "abfss://gold@datalakexperiun.dfs.core.windows.net/"
caminho_arquivo_estadios = f"{caminho_bronze}estadios.csv"


In [0]:
display(dbutils.fs.ls(caminho_bronze))

In [0]:
estadios_df = (spark.read
               .option("encoding", "UTF-16")
               .option("header", "true")
               .option("sep", ",")
               .option("inferSchema", "true")
               .csv(caminho_arquivo_estadios))

In [0]:
display(estadios_df)

In [0]:
estadios_df.printSchema()

In [0]:
estadios_df_renomeado = estadios_df \
    .withColumnRenamed("id", "id_estadio") \
    .withColumnRenamed("name", "nm_estadio") \
    .withColumnRenamed("address", "endereco_estadio") \
    .withColumnRenamed("city", "nm_cidade") \
    .withColumnRenamed("country", "nm_pais") \
    .withColumnRenamed("capacity", "vl_capacidade") \
    .withColumnRenamed("surface", "tp_grama") \
    .withColumnRenamed("image", "link_imagem")

In [0]:
estadios_df_renomeado.display()

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
estadios_df_data = estadios_df_renomeado.withColumn("dt_ingestao", current_timestamp())

In [0]:
display(estadios_df_data)

In [0]:
estadios_df_data.printSchema()

In [0]:
estadios_df_data.write.parquet(f"{caminho_silver}estadios/", mode="overwrite")